# PREPROCESSING AND EXPERIMENT SETUP

Objective:
Evaluate the impact of preprocessing changes:
- Spectral range = 20000 (instead of 10000)
- Binarization = 3 (instead of 5)

We will:
1. Filter species-specific datasets
2. Remove samples with NaNs (spectra or antibiotics)
3. Build LPS patterns
4. Remove patterns with frequency < 10
5. Train:
   - LPS classifier
   - Direct Multilabel classifier
6. 5 random splits (80/20)
7. Report mean ± std for:
   - Weighted F1
   - Hamming Loss
   - Accuracy

## Imports and configuration

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss, accuracy_score
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/DRIAMS_A_AMR_whole_pipeline.pkl"
import pickle

with open(data_path, "rb") as f:
    payload = pickle.load(f)

X = payload["data"]              # spectra matrix (N, D)
y_species = payload["label"]     # species labels (N,)
amr = payload["amr"]             # AMR matrix (N, A)
antibiotics = payload["antibiotics"]  # list length A

print("Dataset loaded.")
print("Spectral data shape:", X.shape)
print("AMR matrix shape:", amr.shape)
print("Number of antibiotics:", len(antibiotics))
print("Unique species:", np.unique(y_species))


Using device: cpu
Dataset loaded.
Spectral data shape: (14925, 6000)
AMR matrix shape: (14925, 9)
Number of antibiotics: 9
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']


In [3]:
species_antibiotics = {
    "Staphylococcus_Aureus": [
        "Oxacillin", "Clindamycin", "Fusidic acid"
    ],
    "Escherichia_Coli": [
        "Ciprofloxacin", "Ceftriaxone",
        "Piperacillin-Tazobactam", "Cefepime"
    ],
    "Klebsiella_Pneumoniae": [
        "Ciprofloxacin", "Ceftriaxone",
        "Imipenem", "Meropenem"
    ],
    "Pseudomonas_Aeruginosa": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ]
}

### MLP Architecture

In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim, activation):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        act_dict = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "logistic": nn.Sigmoid(),
            "identity": nn.Identity()
        }
        
        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(act_dict[activation])
            prev_dim = h
        
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

### Training and evaluation loops

In [6]:
def train_model_with_early_stopping(
    model,
    optimizer,
    criterion,
    X_train,
    y_train,
    device,
    max_epochs=1200,
    patience=50,
    batch_size=128,
    multilabel=False
):
    from torch.utils.data import DataLoader, TensorDataset
    
    # Split train into train/val (10%)
    X_np = X_train.cpu().numpy()
    y_np = y_train.cpu().numpy()
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_np, y_np,
        test_size=0.1,
        random_state=42
    )
    
    X_tr = torch.tensor(X_tr, dtype=torch.float32).to(device)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    
    if multilabel:
        y_tr = torch.tensor(y_tr, dtype=torch.float32).to(device)
    else:
        y_tr = torch.tensor(y_tr, dtype=torch.long).to(device)
    
    dataset = TensorDataset(X_tr, y_tr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    best_state = None
    best_score = -np.inf
    patience_counter = 0
    
    for epoch in range(max_epochs):
        
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
        
        # ---- Validation evaluation ----
        model.eval()
        with torch.no_grad():
            logits_val = model(X_val)
        
        if multilabel:
            probs = torch.sigmoid(logits_val)
            preds = (probs > 0.5).long().cpu().numpy()
            score = evaluate_multilabel_metrics(
                y_val.astype(int), preds
            )["WF1"]
        else:
            preds = torch.argmax(logits_val, dim=1).cpu().numpy()
            score = f1_score(y_val, preds, average="weighted")
        
        if score > best_score:
            best_score = score
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model

In [7]:
from sklearn.metrics import f1_score, accuracy_score, hamming_loss

def evaluate_multilabel_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    n_labels = y_true.shape[1]
    wf1s, accs, hls = [], [], []
    for j in range(n_labels):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        wf1s.append(f1_score(yt, yp, average="weighted"))
        accs.append(accuracy_score(yt, yp))
        hls.append(hamming_loss(yt, yp))
    return {
        "WF1": float(np.mean(wf1s)),
        "ACC": float(np.mean(accs)),
        "HL": float(np.mean(hls)),
    }

def lps_index_to_multilabel(pred_indices: np.ndarray, class_to_pattern: dict) -> np.ndarray:
    preds = []
    for idx in pred_indices:
        pat = class_to_pattern[int(idx)]
        preds.append([int(c) for c in pat])
    return np.asarray(preds, dtype=int)

def evaluate_lps_patterns(y_true_patterns: np.ndarray,
                          y_pred_indices: np.ndarray,
                          class_to_pattern: dict):
    y_true_multi = np.asarray([[int(c) for c in p] for p in y_true_patterns], dtype=int)
    y_pred_multi = lps_index_to_multilabel(y_pred_indices, class_to_pattern)
    return evaluate_multilabel_metrics(y_true_multi, y_pred_multi)

@torch.no_grad()
def predict_lps_class(model, X):
    logits = model(X)
    return torch.argmax(logits, dim=1).cpu().numpy()

@torch.no_grad()
def predict_multilabel(model, X):
    logits = model(X)
    probs = torch.sigmoid(logits)
    return (probs > 0.5).long().cpu().numpy()

### Running experiment

In [8]:
mlp_lps_params = {
    "Escherichia_Coli": dict(
        hidden_layer_sizes=(481, 24, 500),
        activation="relu",
        solver="adam",
        learning_rate_init=2.4378878006413237e-05
    ),
    "Klebsiella_Pneumoniae": dict(
        hidden_layer_sizes=(94, 313, 99),
        activation="tanh",
        solver="adam",
        learning_rate_init=7.693489492906515e-05
    ),
    "Pseudomonas_Aeruginosa": dict(
        hidden_layer_sizes=(402, 208, 435),
        activation="relu",
        solver="adam",
        learning_rate_init=0.004215968282365892
    ),
    "Staphylococcus_Aureus": dict(
        hidden_layer_sizes=(289, 242, 189),
        activation="identity",
        solver="adam",
        learning_rate_init=0.0008078817205716293
    ),
}

mlp_multi_params = {
    "Escherichia_Coli": dict(
        hidden_layer_sizes=(477, 456, 73),
        activation="tanh",
        solver="adam",
        learning_rate_init=1.7708879893278444e-05
    ),
    "Klebsiella_Pneumoniae": dict(
        hidden_layer_sizes=(75, 90, 318),
        activation="tanh",
        solver="adam",
        learning_rate_init=9.459131060843723e-05
    ),
    "Pseudomonas_Aeruginosa": dict(
        hidden_layer_sizes=(463, 472, 327),
        activation="relu",
        solver="adam",
        learning_rate_init=0.002765076022894333
    ),
    "Staphylococcus_Aureus": dict(
        hidden_layer_sizes=(385, 421, 58),
        activation="identity",
        solver="adam",
        learning_rate_init=2.7646086843997938e-05
    ),
}

In [9]:
from sklearn.preprocessing import StandardScaler

results = []

# Convert antibiotics list to numpy for indexing
antibiotics_array = np.array(antibiotics)

for species, ab_list in species_antibiotics.items():

    print(f"\nProcessing {species}")

    # --------------------------
    # 1) Subset by species
    # --------------------------
    species_mask = (y_species == species)
    X_sp = X[species_mask]
    amr_sp = amr[species_mask]

    # --------------------------
    # 2) Subset by antibiotics
    # --------------------------
    ab_indices = [np.where(antibiotics_array == ab)[0][0] for ab in ab_list]
    Y_multi_full = amr_sp[:, ab_indices]

    # --------------------------
    # 3) Drop incomplete cases (NaNs in required antibiotics)
    # --------------------------
    complete_mask = ~np.isnan(Y_multi_full).any(axis=1)
    X_sp = X_sp[complete_mask]
    Y_multi_full = Y_multi_full[complete_mask]

    # --------------------------
    # 4) Build LPS patterns (strings like "0101")
    # --------------------------
    Y_multi_int = Y_multi_full.astype(int)
    LPS_patterns = np.array(["".join(row.astype(str)) for row in Y_multi_int], dtype=object)

    # --------------------------
    # 5) Remove rare patterns (<10)
    # --------------------------
    pattern_counts = Counter(LPS_patterns)
    valid_patterns = {p for p, c in pattern_counts.items() if c >= 10}

    valid_mask = np.array([p in valid_patterns for p in LPS_patterns])
    X_sp = X_sp[valid_mask]
    Y_multi_full = Y_multi_full[valid_mask]
    LPS_patterns = LPS_patterns[valid_mask]

    metrics_lps = []
    metrics_multi = []

    # --------------------------
    # 6) 5 random splits (80/20)
    # --------------------------
    for seed in range(5):

        X_train, X_test, y_train_multi, y_test_multi, pat_train, pat_test = train_test_split(
            X_sp, Y_multi_full, LPS_patterns,
            test_size=0.2,
            random_state=seed,
            stratify=LPS_patterns
        )

        # --------------------------
        # 7) LPS mapping
        # --------------------------
        patterns_train_unique = np.unique(pat_train)
        pattern_to_class = {p: i for i, p in enumerate(patterns_train_unique)}
        class_to_pattern = {i: p for p, i in pattern_to_class.items()}

        y_train_lps = np.array([pattern_to_class[p] for p in pat_train], dtype=int)

        # Drop test samples with unseen patterns (should be rare due to stratify, but safe)
        seen_mask_test = np.array([p in pattern_to_class for p in pat_test])
        X_test = X_test[seen_mask_test]
        y_test_multi = y_test_multi[seen_mask_test]
        pat_test = pat_test[seen_mask_test]
        y_test_lps = np.array([pattern_to_class[p] for p in pat_test], dtype=int)

        n_classes = len(pattern_to_class)

        # --------------------------
        # 8) StandardScaler
        # --------------------------
        #scaler = StandardScaler()
        #X_train = scaler.fit_transform(X_train).astype(np.float32)
        #X_test = scaler.transform(X_test).astype(np.float32)

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        # --------------------------
        # 9) LPS model (multiclass)
        # --------------------------
        model_lps = MLP(
            input_dim=X_sp.shape[1],
            hidden_layers=mlp_lps_params[species]["hidden_layer_sizes"],
            output_dim=n_classes,
            activation=mlp_lps_params[species]["activation"]
        ).to(device)

        optimizer = optim.Adam(
            model_lps.parameters(),
            lr=mlp_lps_params[species]["learning_rate_init"]
        )

        criterion = nn.CrossEntropyLoss()

        y_train_lps_t = torch.tensor(y_train_lps, dtype=torch.long).to(device)

        model_lps = train_model_with_early_stopping(
            model_lps,optimizer,criterion, X_train_t, y_train_lps_t, device,multilabel=False)
        pred_class = predict_lps_class(model_lps, X_test_t)  # uses argmax on logits
        lps_metrics = evaluate_lps_patterns(pat_test, pred_class, class_to_pattern)  # paper-style
        metrics_lps.append(lps_metrics)

        # --------------------------
        # 10) Direct multilabel model, evaluated paper-style (per-antibiotic mean)
        # --------------------------
        model_multi = MLP(
            input_dim=X_sp.shape[1],
            hidden_layers=mlp_multi_params[species]["hidden_layer_sizes"],
            output_dim=len(ab_list),
            activation=mlp_multi_params[species]["activation"]
        ).to(device)

        optimizer = optim.Adam(
            model_multi.parameters(),
            lr=mlp_multi_params[species]["learning_rate_init"]
        )

        criterion = nn.BCEWithLogitsLoss()

        y_train_multi_t = torch.tensor(y_train_multi, dtype=torch.float32).to(device)

        model_multi = train_model_with_early_stopping(
			model_multi,
			optimizer,
			criterion,
			X_train_t,
			y_train_multi_t,
			device,
			multilabel=True
		)
        pred_multi = predict_multilabel(model_multi, X_test_t)  # sigmoid + 0.5 threshold
        multi_metrics = evaluate_multilabel_metrics(y_test_multi.astype(int), pred_multi.astype(int))  # paper-style
        metrics_multi.append(multi_metrics)

    df_lps = pd.DataFrame(metrics_lps)
    df_multi = pd.DataFrame(metrics_multi)

    results.append({
        "Species": species,
        "LPS_WF1": f"{df_lps['WF1'].mean():.3f} ± {df_lps['WF1'].std():.3f}",
        "LPS_HL": f"{df_lps['HL'].mean():.3f} ± {df_lps['HL'].std():.3f}",
        "LPS_ACC": f"{df_lps['ACC'].mean():.3f} ± {df_lps['ACC'].std():.3f}",
        "Multi_WF1": f"{df_multi['WF1'].mean():.3f} ± {df_multi['WF1'].std():.3f}",
        "Multi_HL": f"{df_multi['HL'].mean():.3f} ± {df_multi['HL'].std():.3f}",
        "Multi_ACC": f"{df_multi['ACC'].mean():.3f} ± {df_multi['ACC'].std():.3f}",
    })


Processing Staphylococcus_Aureus

Processing Escherichia_Coli

Processing Klebsiella_Pneumoniae

Processing Pseudomonas_Aeruginosa


In [10]:
results_df = pd.DataFrame(results)
results_df

,Species,LPS_WF1,LPS_HL,LPS_ACC,Multi_WF1,Multi_HL,Multi_ACC
0,Staphylococcus_Aureus,0.889 ± 0.007,0.105 ± 0.006,0.895 ± 0.006,0.889 ± 0.006,0.104 ± 0.006,0.896 ± 0.006
1,Escherichia_Coli,0.846 ± 0.007,0.147 ± 0.008,0.853 ± 0.008,0.842 ± 0.004,0.151 ± 0.005,0.849 ± 0.005
2,Klebsiella_Pneumoniae,0.920 ± 0.007,0.074 ± 0.005,0.926 ± 0.005,0.919 ± 0.007,0.074 ± 0.006,0.926 ± 0.006
3,Pseudomonas_Aeruginosa,0.849 ± 0.011,0.134 ± 0.010,0.866 ± 0.010,0.851 ± 0.008,0.132 ± 0.004,0.868 ± 0.004
